In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
#Reading the data
df = pd.read_csv('Dataset_Day7.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,33.6,0.627,50,1
1,1,85,66,26.6,0.351,31,0
2,8,183,64,23.3,0.672,32,1
3,1,89,66,28.1,0.167,21,0
4,0,137,40,43.1,2.288,33,1


In [4]:
#Replacing 0 with NaN for the required columns
cols = ["Glucose", "BloodPressure", "BMI", "DiabetesPedigreeFunction"]
df[cols] = df[cols].replace(0, np.nan)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   763 non-null    float64
 2   BloodPressure             733 non-null    float64
 3   BMI                       757 non-null    float64
 4   DiabetesPedigreeFunction  768 non-null    float64
 5   Age                       768 non-null    int64  
 6   Outcome                   768 non-null    int64  
dtypes: float64(4), int64(3)
memory usage: 42.1 KB


In [5]:
#Treating missing data
numeric_col=[]

for col in df.columns:
    if df[col].dtype.name != 'object':
        numeric_col.append(col)

print(numeric_col)

median_value = df[numeric_col].median()
print(median_value)

df = df.fillna(median_value)
df.info()

['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
Pregnancies                   3.0000
Glucose                     117.0000
BloodPressure                72.0000
BMI                          32.3000
DiabetesPedigreeFunction      0.3725
Age                          29.0000
Outcome                       0.0000
dtype: float64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 7 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    float64
 2   BloodPressure             768 non-null    float64
 3   BMI                       768 non-null    float64
 4   DiabetesPedigreeFunction  768 non-null    float64
 5   Age                       768 non-null    int64  
 6   Outcome                   768 non-null    int64  
dtypes: float64(4), int64(3)
m

In [6]:
for col in cols:
    df[col] = (df[col] - df[col].mean()) / df[col].std()

# Outlier Detection using z-score
OutlierRows = df[
    (df["Glucose"] > 3) | (df["Glucose"] < -3) |
    (df["BloodPressure"] > 3) | (df["BloodPressure"] < -3) |
    (df["BMI"] > 3) | (df["BMI"] < -3) |
    (df["DiabetesPedigreeFunction"] > 3) | (df["DiabetesPedigreeFunction"] < -3)
]

print("% of Outlier rows in the dataset is " + str(len(OutlierRows) / len(df) * 100) + "\n")

data_OutlierFree = df.drop(OutlierRows.index, axis=0)

% of Outlier rows in the dataset is 2.734375



In [7]:
# Data Splitting into 80% training and 20% testing data

X = data_OutlierFree.drop("Outcome",axis=1)
y = data_OutlierFree["Outcome"]

from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=99)
len(X_train),len(X_test)

(597, 150)

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import GaussianNB

# Fit a Decision Tree model as comparison

nb = GaussianNB() ## try out other models on new data for your own research
nb = nb.fit(X_train, y_train)
y_pred = nb.predict(X_test)

print("Model Performance metrics are as below :-\n")
print("Accuracy is "+str(accuracy_score(y_test,y_pred)))
print("Precision is "+str(precision_score(y_test,y_pred)))
print("Recall is "+str(recall_score(y_test,y_pred)))
print("F1-Score is "+str(f1_score(y_test,y_pred)))

Model Performance metrics are as below :-

Accuracy is 0.8
Precision is 0.6382978723404256
Recall is 0.6976744186046512
F1-Score is 0.6666666666666666


Observations:-

Accuracy of 80% indicates that the model is genrally performing well.
However, Precision of 63.8% indicates a higher false positive prediction rate.
Recall of 69.8% indicates that the model is better at detecting true positives than minimizing false positives.
F1-score of 66.7% indicates a good balance between Precision and Recall.

In [9]:
from sklearn.model_selection import cross_val_score, cross_val_predict, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.naive_bayes import GaussianNB

# Define features and target
X = data_OutlierFree.drop("Outcome", axis=1)
y = data_OutlierFree["Outcome"]

# Create the model
nb = GaussianNB()

# Define the k-Fold Cross Validator (k=10)
kf = KFold(n_splits=10, shuffle=True, random_state=99)

# Perform cross-validated predictions
y_pred_cv = cross_val_predict(nb, X, y, cv=kf)

acc = accuracy_score(y, y_pred_cv)
prec = precision_score(y, y_pred_cv)
rec = recall_score(y, y_pred_cv)
f1 = f1_score(y, y_pred_cv)

print("Model Performance with 10-Fold Cross Validation:\n")
print("Accuracy  :", acc)
print("Precision :", prec)
print("Recall    :", rec)
print("F1-Score  :", f1)


Model Performance with 10-Fold Cross Validation:

Accuracy  : 0.7603748326639893
Precision : 0.6652892561983471
Recall    : 0.6216216216216216
F1-Score  : 0.6427145708582834


Accuracy is 76%, which is good but F1-score is much lower i.e., 64.3% which indicates uneven performance.
Precision is 66.5%, which Recall is 62.2%, which means that the while the model predicts fewer positive cases, it's likely that those predictions are correct.
F1-score shows a balance between Precision and Recall

In [10]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_predict, KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Define base model (weak learner)
base_model = DecisionTreeClassifier(max_depth=1, random_state=99)

# Create AdaBoost model
ada_model = AdaBoostClassifier(estimator=base_model, n_estimators=50, learning_rate=1.0, random_state=42)

# 10-fold Cross Validation setup
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# Cross-validated predictions
y_pred_ada = cross_val_predict(ada_model, X, y, cv=kf)

# Evaluate performance
acc = accuracy_score(y, y_pred_ada)
prec = precision_score(y, y_pred_ada)
rec = recall_score(y, y_pred_ada)
f1 = f1_score(y, y_pred_ada)

# Display results
print("AdaBoost Performance with 10-Fold Cross Validation:\n")
print("Accuracy  :", acc)
print("Precision :", prec)
print("Recall    :", rec)
print("F1-Score  :", f1)

AdaBoost Performance with 10-Fold Cross Validation:

Accuracy  : 0.7550200803212851
Precision : 0.6696428571428571
Recall    : 0.5791505791505791
F1-Score  : 0.6211180124223602


Observations:-

Accuracy was slightly reduced in Adaboost (75.5%) from Naive Bayes classifier (76.0%).
Model is more careful with false positives (66.9% Precision as compared to 66.5%).
Recall is much lower i.e., 57.9% indicating higher false negatives.
F1-score is lower i.e., 62.1%.
Adaboost gives decent performance, improving Precision, but it may not be the best for this case.